# Silver Transform - Ejecución Manual
Notebook de respaldo para la transformación Silver en caso de fallo en Airflow.
Lee datos desde Bronze (Parquet) y los escribe en Silver (Iceberg) usando merge manual.

## 1. Configuración e imports

In [1]:
import os
import pyspark
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import col, current_date, regexp_replace, trim
from pyspark.sql.types import IntegerType

# FIX: el SDK de AWS requiere una región aunque sea ficticia para MinIO
os.environ["AWS_REGION"]              = "us-east-1"
os.environ["AWS_DEFAULT_REGION"]      = "us-east-1"
os.environ["AWS_ACCESS_KEY_ID"]       = "admin"
os.environ["AWS_SECRET_ACCESS_KEY"]   = "password"

In [ ]:
# ── Parámetros de conexión ──────────────────────────────────────────────────
CATALOG_URI    = "http://nessie:19120/api/v1"
WAREHOUSE      = "s3a://proyecto2/silver"
S3_ENDPOINT    = "http://minio:9000"
AWS_ACCESS_KEY = "admin"
AWS_SECRET_KEY = "password"

BRONZE_BASE = "s3a://proyecto2/bronze"

USERS_PATH         = f"{BRONZE_BASE}/users/*.parquet"
COMMENTS_2023_PATH = f"{BRONZE_BASE}/comments_2023/*.parquet"
COMMENTS_2024_PATH = f"{BRONZE_BASE}/comments_2024/*.parquet"
POSTS_2023_PATH    = f"{BRONZE_BASE}/posts_2023/*.parquet"
POSTS_2024_PATH    = f"{BRONZE_BASE}/posts_2024/*.parquet"

SILVER_NAMESPACE = "nessie.silver"
USERS_TABLE    = f"{SILVER_NAMESPACE}.users_hist"
COMMENTS_TABLE = f"{SILVER_NAMESPACE}.comments_hist"
POSTS_TABLE    = f"{SILVER_NAMESPACE}.posts_hist"

#USERS_LIMIT    = 1000
#COMMENTS_LIMIT = 1000
#POSTS_LIMIT    = 200

## 2. Inicializar SparkSession

In [3]:
conf = (
    pyspark.SparkConf()
    .setAppName("silver_transform_notebook")
    .set("spark.driver.memory", "8g")
    .set("spark.executor.memory", "16g")
    .set("spark.sql.shuffle.partitions", "400")
    .set("spark.driver.maxResultSize", "4g")
    .set("spark.network.timeout", "800s")
    .set("spark.executor.heartbeatInterval", "60s")
    .set("spark.sql.broadcastTimeout", "1200")
    .set("spark.sql.autoBroadcastJoinThreshold", "104857600")
    .set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "268435456")
    .set("spark.sql.adaptive.enabled", "true")
    .set("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .set(
        "spark.jars.packages",
        ",".join([
            "org.postgresql:postgresql:42.7.3",
            "org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.5.0",
            "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.4_2.12:0.77.1",
            "software.amazon.awssdk:bundle:2.24.8",
            "software.amazon.awssdk:url-connection-client:2.24.8",
            "org.apache.hadoop:hadoop-aws:3.2.0",
        ]),
    )
    .set(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
        "org.projectnessie.spark.extensions.NessieSparkSessionExtensions",
    )
    # Catálogo Nessie
    .set("spark.sql.catalog.nessie",                          "org.apache.iceberg.spark.SparkCatalog")
    .set("spark.sql.catalog.nessie.uri",                      CATALOG_URI)
    .set("spark.sql.catalog.nessie.ref",                      "main")
    .set("spark.sql.catalog.nessie.authentication.type",      "NONE")
    .set("spark.sql.catalog.nessie.catalog-impl",             "org.apache.iceberg.nessie.NessieCatalog")
    .set("spark.sql.catalog.nessie.warehouse",                WAREHOUSE)
    .set("spark.sql.catalog.nessie.io-impl",                  "org.apache.iceberg.aws.s3.S3FileIO")
    .set("spark.sql.catalog.nessie.s3.endpoint",              S3_ENDPOINT)
    .set("spark.sql.catalog.nessie.s3.path-style-access",     "true")
    .set("spark.sql.catalog.nessie.s3.access-key-id",         AWS_ACCESS_KEY)
    .set("spark.sql.catalog.nessie.s3.secret-access-key",     AWS_SECRET_KEY)
    # FIX: región requerida por el SDK de AWS aunque MinIO no la use
    .set("spark.sql.catalog.nessie.s3.region",                "us-east-1")
    # Hadoop S3A
    .set("spark.hadoop.fs.s3a.endpoint",                      S3_ENDPOINT)
    .set("spark.hadoop.fs.s3a.access.key",                    AWS_ACCESS_KEY)
    .set("spark.hadoop.fs.s3a.secret.key",                    AWS_SECRET_KEY)
    .set("spark.hadoop.fs.s3a.path.style.access",             "true")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider",      "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .set("spark.hadoop.fs.s3a.impl",                          "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3a.connection.ssl.enabled",        "false")
    # FIX: región para el cliente S3 del SDK de AWS
    .set("spark.hadoop.fs.s3a.endpoint.region",               "us-east-1")
    .set("spark.driver.extraJavaOptions",
         "-Daws.region=us-east-1 -Daws.accessKeyId=admin -Daws.secretAccessKey=password")
    .set("spark.executor.extraJavaOptions",
         "-Daws.region=us-east-1 -Daws.accessKeyId=admin -Daws.secretAccessKey=password")
)

spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("SparkSession iniciada correctamente.")
spark

SparkSession iniciada correctamente.


## 3. Funciones auxiliares

In [4]:
def create_namespace_if_needed():
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {SILVER_NAMESPACE}")
    print(f"Namespace {SILVER_NAMESPACE} listo.")


def spark_table_exists(table_name: str) -> bool:
    try:
        return spark.catalog.tableExists(table_name)
    except Exception:
        return False


def merge_into_iceberg(df_new: DataFrame, table_name: str, key_col: str = "id") -> None:
    """
    MERGE manual con PySpark puro.
    Equivalente a: MERGE WHEN MATCHED UPDATE / WHEN NOT MATCHED INSERT.
    Evita el bug de SQL MERGE en Iceberg 1.5 + Nessie 0.77.
    """
    if not spark_table_exists(table_name):
        print(f"La tabla {table_name} no existe. Creando con carga inicial...")
        (
            df_new.writeTo(table_name)
            .using("iceberg")
            .tableProperty("format-version", "2")
            .create()
        )
        print(f"Tabla {table_name} creada correctamente.")
        return

    print(f"Leyendo tabla existente {table_name}...")
    df_existing = spark.read.format("iceberg").load(table_name)

    # Registros que NO vienen en el nuevo batch: se conservan tal cual
    df_unchanged = df_existing.join(
        df_new.select(key_col),
        on=key_col,
        how="left_anti"
    )

    # Union: sin cambios + nuevo batch (cubre updates e inserts)
    df_merged = df_unchanged.unionByName(df_new, allowMissingColumns=True)

    print(f"Sobreescribiendo {table_name} con datos mergeados...")
    (
        df_merged.writeTo(table_name)
        .using("iceberg")
        .overwritePartitions()
    )
    print(f"MERGE completado en {table_name}.")

In [5]:
def transform_users(df: DataFrame) -> DataFrame:
    return (
        df.withColumn("display_name", col("display_name").cast("string"))
          .withColumn("location",     col("location").cast("string"))
          .withColumn("about_me",     col("about_me").cast("string"))
          .withColumn("website_url",  col("website_url").cast("string"))
          .withColumn("reputation",   col("reputation").cast("string").cast(IntegerType()))
          .withColumn("fecha_cargue", current_date())
    )


def transform_comments(df: DataFrame) -> DataFrame:
    return (
        df.withColumn("text",              col("text").cast("string"))
          .withColumn("user_display_name", col("user_display_name").cast("string"))
          .withColumn("fecha_cargue",      current_date())
    )


def transform_posts(df: DataFrame) -> DataFrame:
    df = (
        df.withColumn("body",            col("body").cast("string"))
          .withColumn("title",           col("title").cast("string"))
          .withColumn("tags",            col("tags").cast("string"))
          .withColumn("content_license", col("content_license").cast("string"))
          .withColumn("parent_id",       col("parent_id").cast("string").cast(IntegerType()))
    )
    df = df.withColumn("body", regexp_replace(col("body"), "<[^>]*>", ""))
    df = df.withColumn("tags", regexp_replace(col("tags"), r"\|", ","))
    df = df.withColumn("tags", regexp_replace(trim(col("tags")), r"^,|,$", ""))
    df = df.withColumn("fecha_cargue", current_date())
    return df

## 4. Crear namespace Silver

In [6]:
create_namespace_if_needed()

Namespace nessie.silver listo.


## 5. Silver — Users

In [7]:
print("Leyendo users desde Bronze...")
df_users = spark.read.parquet(USERS_PATH).limit(USERS_LIMIT)
df_users = transform_users(df_users)
print(f"Registros a cargar: {df_users.count()}")
df_users.printSchema()

Leyendo users desde Bronze...
Registros a cargar: 1000
root
 |-- id: long (nullable = true)
 |-- reputation: integer (nullable = true)
 |-- creation_date: timestamp (nullable = true)
 |-- display_name: string (nullable = true)
 |-- last_access_date: timestamp (nullable = true)
 |-- about_me: string (nullable = true)
 |-- views: long (nullable = true)
 |-- up_votes: long (nullable = true)
 |-- down_votes: long (nullable = true)
 |-- website_url: string (nullable = true)
 |-- location: string (nullable = true)
 |-- account_id: long (nullable = true)
 |-- fecha_cargue: date (nullable = false)



In [8]:
merge_into_iceberg(df_users, USERS_TABLE, key_col="id")
print("Silver users completado.")

Leyendo tabla existente nessie.silver.users_hist...
Sobreescribiendo nessie.silver.users_hist con datos mergeados...
MERGE completado en nessie.silver.users_hist.
Silver users completado.


In [9]:
spark.read.format("iceberg").load(USERS_TABLE).show(5)

+-------+----------+--------------------+-------------+--------------------+--------+-----+--------+----------+--------------------+-------------+----------+------------+
|     id|reputation|       creation_date| display_name|    last_access_date|about_me|views|up_votes|down_votes|         website_url|     location|account_id|fecha_cargue|
+-------+----------+--------------------+-------------+--------------------+--------+-----+--------+----------+--------------------+-------------+----------+------------+
|2740198|         1|2013-09-02 14:33:...|        McKee|2015-07-01 14:19:...|        |    0|       0|         0|                    |             |   3249498|  2026-04-02|
|2740199|         1|2013-09-02 14:33:...|  Zaeka Baeka|2016-07-08 06:56:...|        |    0|       0|         0|                    |             |   3249499|  2026-04-02|
|2740200|         5|2013-09-02 14:33:...|Daniel Thomas|2013-10-31 11:19:...|        |    5|       0|         0|                    |             

## 6. Silver — Comments

In [10]:
print("Leyendo comments desde Bronze...")
df_comments_2023 = spark.read.parquet(COMMENTS_2023_PATH).limit(COMMENTS_LIMIT)
df_comments_2024 = spark.read.parquet(COMMENTS_2024_PATH).limit(COMMENTS_LIMIT)
df_comments_2023 = transform_comments(df_comments_2023)
df_comments_2024 = transform_comments(df_comments_2024)
df_comments = df_comments_2023.unionByName(df_comments_2024, allowMissingColumns=True)
print(f"Registros a cargar: {df_comments.count()}")
df_comments.printSchema()

Leyendo comments desde Bronze...
Registros a cargar: 2000
root
 |-- id: long (nullable = true)
 |-- post_id: long (nullable = true)
 |-- score: long (nullable = true)
 |-- text: string (nullable = true)
 |-- creation_date: timestamp (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_display_name: string (nullable = true)
 |-- fecha_cargue: date (nullable = false)



In [11]:
merge_into_iceberg(df_comments, COMMENTS_TABLE, key_col="id")
print("Silver comments completado.")

Leyendo tabla existente nessie.silver.comments_hist...
Sobreescribiendo nessie.silver.comments_hist con datos mergeados...
MERGE completado en nessie.silver.comments_hist.
Silver comments completado.


In [12]:
spark.read.format("iceberg").load(COMMENTS_TABLE).show(5)

+---------+--------+-----+--------------------+--------------------+--------+-----------------+------------+
|       id| post_id|score|                text|       creation_date| user_id|user_display_name|fecha_cargue|
+---------+--------+-----+--------------------+--------------------+--------+-----------------+------------+
|135792707|55603848|    0|See https://stack...|2023-09-02 16:12:...|  483622|                 |  2026-04-02|
|133693017|55604807|    0|AbhijeetKhangarot...|2023-03-20 11:28:...| 5695838|                 |  2026-04-02|
|133779669|55607611|    0|This old answer w...|2023-03-25 15:12:...| 2876079|                 |  2026-04-02|
|134438307|55608511|    0|Thank you good si...|2023-05-12 09:52:...| 4672736|                 |  2026-04-02|
|134784432|55609573|    0|It does not allow...|2023-06-09 11:03:...|14062144|                 |  2026-04-02|
+---------+--------+-----+--------------------+--------------------+--------+-----------------+------------+
only showing top 5 

## 7. Silver — Posts

In [13]:
print("Leyendo posts desde Bronze...")
df_posts_2023 = spark.read.parquet(POSTS_2023_PATH).limit(POSTS_LIMIT)
df_posts_2024 = spark.read.parquet(POSTS_2024_PATH).limit(POSTS_LIMIT)
df_posts_2023 = transform_posts(df_posts_2023)
df_posts_2024 = transform_posts(df_posts_2024)
df_posts = df_posts_2023.unionByName(df_posts_2024, allowMissingColumns=True)
print(f"Registros a cargar: {df_posts.count()}")
df_posts.printSchema()

Leyendo posts desde Bronze...
Registros a cargar: 400
root
 |-- id: long (nullable = true)
 |-- post_type_id: long (nullable = true)
 |-- accepted_answer_id: long (nullable = true)
 |-- creation_date: timestamp (nullable = true)
 |-- score: long (nullable = true)
 |-- view_count: long (nullable = true)
 |-- body: string (nullable = true)
 |-- owner_user_id: long (nullable = true)
 |-- owner_display_name: binary (nullable = true)
 |-- last_editor_user_id: long (nullable = true)
 |-- last_editor_display_name: binary (nullable = true)
 |-- last_edit_date: timestamp (nullable = true)
 |-- last_activity_date: timestamp (nullable = true)
 |-- title: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- answer_count: long (nullable = true)
 |-- comment_count: long (nullable = true)
 |-- favorite_count: long (nullable = true)
 |-- content_license: string (nullable = true)
 |-- parent_id: integer (nullable = true)
 |-- community_owned_date: timestamp (nullable = true)
 |-- closed_da

In [14]:
merge_into_iceberg(df_posts, POSTS_TABLE, key_col="id")
print("Silver posts completado.")

Leyendo tabla existente nessie.silver.posts_hist...
Sobreescribiendo nessie.silver.posts_hist con datos mergeados...
MERGE completado en nessie.silver.posts_hist.
Silver posts completado.


In [15]:
spark.read.format("iceberg").load(POSTS_TABLE).show(5)

+--------+------------+------------------+--------------------+-----+----------+--------------------+-------------+------------------+-------------------+------------------------+--------------------+--------------------+--------------------+--------------------+------------+-------------+--------------+---------------+---------+--------------------+-------------------+------------+
|      id|post_type_id|accepted_answer_id|       creation_date|score|view_count|                body|owner_user_id|owner_display_name|last_editor_user_id|last_editor_display_name|      last_edit_date|  last_activity_date|               title|                tags|answer_count|comment_count|favorite_count|content_license|parent_id|community_owned_date|        closed_date|fecha_cargue|
+--------+------------+------------------+--------------------+-----+----------+--------------------+-------------+------------------+-------------------+------------------------+--------------------+--------------------+-------

## 8. Resumen final

In [16]:
print("=" * 50)
print("RESUMEN SILVER")
print("=" * 50)
for table in [USERS_TABLE, COMMENTS_TABLE, POSTS_TABLE]:
    try:
        count = spark.read.format("iceberg").load(table).count()
        print(f"  {table}: {count} registros")
    except Exception as e:
        print(f"  {table}: ERROR - {e}")
print("=" * 50)
print("Silver transform completado correctamente.")

RESUMEN SILVER
  nessie.silver.users_hist: 2000 registros
  nessie.silver.comments_hist: 4000 registros
  nessie.silver.posts_hist: 1000 registros
Silver transform completado correctamente.
